## Regridding data from one grid to another for Initial Conditions using xESMF

What follows an example of how to use xESMF to regrid data (mostly from [roocs](https://github.com/roocs)). The objective is to regrid a data set from one grid to another for the use of initial conditions. Usually my go to tool is SOSIE/SCRIP to regrid data, but this is a pythonic version.

In [8]:
import os; os.environ['PROJ_LIB'] = '/work/benbar/envs/pyic/' # avoid basemap import error

In [9]:
%matplotlib inline
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import xarray as xr
import cf_xarray as cfxr
import xesmf as xe
import scipy.sparse as sps
import clisops.core as clore
import clisops as cl
import textwrap
import math
import copy as cp
import numpy as np
import warnings
import dask.array as da
import gsw
import scipy.ndimage as ndimage

from scipy.spatial import cKDTree
from scipy.ndimage import maximum_filter
from numba import njit, prange

xr.set_options(display_style='html');

### Functions for regrdidding

In [10]:
def compute_index_map_kdtree(
    fine_ds: xr.Dataset, #| xr.DataArray,
    coarse_ds: xr.Dataset, #| xr.DataArray,
    lon_name: str = "lon",
    lat_name: str = "lat"
) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute fine->coarse index maps for arbitrary curvilinear grids
    using nearest-neighbor search with KDTree.

    Args:
        fine_ds (xr.Dataset | xr.DataArray): Fine grid dataset or dataarray with lon/lat.
        coarse_ds (xr.Dataset | xr.DataArray): Coarse grid dataset or dataarray with lon/lat.
        lon_name (str): Name of longitude variable in datasets. Default "lon".
        lat_name (str): Name of latitude variable in datasets. Default "lat".

    Returns:
        tuple[np.ndarray, np.ndarray]: 
            - j_map (ndarray): Coarse j-index for each fine cell
            - i_map (ndarray): Coarse i-index for each fine cell
    """

    # --- Type checks ---
    if not isinstance(fine_ds, (xr.Dataset, xr.DataArray)):
        raise TypeError(f"fine_ds must be xarray.Dataset or xarray.DataArray, got {type(fine_ds)}")
    if not isinstance(coarse_ds, (xr.Dataset, xr.DataArray)):
        raise TypeError(f"coarse_ds must be xarray.Dataset or xarray.DataArray, got {type(coarse_ds)}")

    # --- Variable existence checks ---
    for name, ds in [(lon_name, fine_ds), (lat_name, fine_ds),
                     (lon_name, coarse_ds), (lat_name, coarse_ds)]:
        if name not in ds:
            raise KeyError(f"Variable '{name}' not found in dataset: {list(ds.variables)}")

    # --- Extract and squeeze coordinates ---
    fine_lon = np.squeeze(fine_ds[lon_name].values)
    fine_lat = np.squeeze(fine_ds[lat_name].values)
    coarse_lon = np.squeeze(coarse_ds[lon_name].values)
    coarse_lat = np.squeeze(coarse_ds[lat_name].values)

    # --- Shape checks ---
    if fine_lon.ndim != 2 or fine_lat.ndim != 2:
        raise ValueError(f"Fine grid lon/lat must be 2D after squeeze, got shapes {fine_lon.shape} and {fine_lat.shape}")
    if coarse_lon.ndim != 2 or coarse_lat.ndim != 2:
        raise ValueError(f"Coarse grid lon/lat must be 2D after squeeze, got shapes {coarse_lon.shape} and {coarse_lat.shape}")

    # --- Dtype checks ---
    if not np.issubdtype(fine_lon.dtype, np.floating) or not np.issubdtype(fine_lat.dtype, np.floating):
        raise TypeError("Fine grid lon/lat must be float arrays.")
    if not np.issubdtype(coarse_lon.dtype, np.floating) or not np.issubdtype(coarse_lat.dtype, np.floating):
        raise TypeError("Coarse grid lon/lat must be float arrays.")

    # --- KDTree nearest neighbor mapping ---
    ny_c, nx_c = coarse_lon.shape
    coarse_points = np.column_stack((coarse_lon.ravel(), coarse_lat.ravel()))
    fine_points = np.column_stack((fine_lon.ravel(), fine_lat.ravel()))

    tree = cKDTree(coarse_points)
    _, idx = tree.query(fine_points)

    j_map = (idx // nx_c).reshape(fine_lon.shape)
    i_map = (idx % nx_c).reshape(fine_lon.shape)

    return j_map, i_map

def add_tmask_from_field(
    field: xr.DataArray,
    grid_ds: xr.Dataset,
    missing_value: float
) -> xr.Dataset:
    """
    Create a binary land-sea mask (tmask) from a 4D DataArray 
    (time_counter, deptht, y, x) and append it to a grid Dataset.

    Parameters
    ----------
    field : xr.DataArray
        4D DataArray with dims (time_counter, deptht, y, x).
        Missing/land points should equal `missing_value`.
    grid_ds : xr.Dataset
        Dataset containing grid information. Will be returned with `tmask`.
    missing_value : float
        Value representing missing/land points in `field`.

    Returns
    -------
    xr.Dataset
        Copy of `grid_ds` with a 3D (deptht, y, x) DataArray `tmask` added.
    """

    # --- Type and dimension checks ---
    if not isinstance(field, xr.DataArray):
        raise TypeError(f"`field` must be an xarray.DataArray, got {type(field)}")
    required_dims = {"time_counter", "deptht", "y", "x"}
    if not required_dims.issubset(field.dims):
        raise ValueError(f"`field` must have dims {required_dims}, got {field.dims}")

    # --- Drop time dimension (assume static mask over time) ---
    data_3d = field.isel(time_counter=0)

    # --- Create binary mask: 1 for ocean, 0 for land ---
    mask = xr.where(data_3d != missing_value, 1, 0).astype(np.int8)
    mask = mask.drop_vars('time_counter')
    mask.name = "tmask"
    mask.attrs.update({
        "long_name": "Land-sea mask",
        "description": "1 = sea, 0 = land",
        "missing_value": 0,
    })

    # --- Attach to grid dataset ---
    out_ds = grid_ds.copy()
    out_ds["tmask"] = mask

    return out_ds

def coarse_from_fine_indices(fine_mask, fine_to_coarse_j, fine_to_coarse_i, coarse_shape):
    """
    Map a fine mask to coarse cells using fast index binning.

    Args:
        fine_mask (2D ndarray): Boolean ocean mask on fine grid.
        fine_to_coarse_j, fine_to_coarse_i (2D ndarray): Index maps from fine -> coarse.
        coarse_shape (tuple): Shape of coarse grid (ny, nx).

    Returns:
        ndarray: Coarse mask (1 if any fine cell is wet, else 0).
    """
    ny, nx = coarse_shape
    coarse_flag = np.zeros((ny, nx), dtype=int)

    flat_mask = fine_mask.ravel()
    flat_j = fine_to_coarse_j.ravel()
    flat_i = fine_to_coarse_i.ravel()

    # Only process wet cells
    wet_idx = np.where(flat_mask > 0)[0]
    np.add.at(coarse_flag, (flat_j[wet_idx], flat_i[wet_idx]), 1)

    return coarse_flag > 0

def add_tmask_domcfg(
    ds: xr.Dataset, 
) -> xr.Dataset:
    
    """
    Function to generate tmask from top_level and bottom_level and 
    add to the Dataset.
    Parameters
    ----------
    ds: Dataset
        xarray dataset containing coordinate information
    Returns
    -------
    Dataset
        updated xarray dataset with tmask variable
    """   

    # check that dims are x,y,z
    
    # generate tmask
    ds['tmask'] = xr.zeros_like(ds.e3t_0)
    for jk in range(ds.nav_lev.size):
        ds.tmask[jk,:,:] = xr.where(ds.bottom_level.squeeze()>jk, 1, 0)
        
    # TODO: adding depth range adjustment
    
    return ds

def e3_to_depth(pe3t, pe3w, jpk):
    '''
    funtion e3_to_depth
    '''
    
    ds_dims = pe3t.dims
    
    if 'nav_lev' in ds_dims:
        pe3t = pe3t.rename({'nav_lev': 'z'})
        pe3w = pe3w.rename({'nav_lev': 'z'})
        
    pdept      = xr.zeros_like(pe3t).rename('gdept')
    pdepw      = xr.zeros_like(pe3t).rename('gdepw')
      
    pdept = pe3w.cumsum('z') - (0.5 * pe3w.values[0:1, :, :])
    pdepw = pe3t.cumsum('z') - pe3t.values
            
    return pdept, pdepw

def coarse_from_fine_indices3(
    fine_mask: xr.DataArray,       # (zf, yf, xf), 1=wet, 0=land
    fine_depth: xr.DataArray,      # (zf, yf, xf)
    coarse_depth: xr.DataArray,    # (zc, yc, xc)
    jmap: np.ndarray,              # (yf, xf), coarse y indices
    imap: np.ndarray               # (yf, xf), coarse x indices
) -> xr.DataArray:
    """
    Build a 3D coarse wet mask: coarse_flag[zc, yc, xc] = 1 if ANY mapped
    fine column has depth deeper than that coarse level.

    Args:
        fine_mask    : Fine mask (1=wet, 0=land)
        fine_depth   : Depth of fine levels (same shape as fine_mask)
        coarse_depth : Depth of coarse levels (zc, yc, xc)
        jmap, imap   : Arrays mapping each fine (y,x) cell to a coarse (y,x) index

    Returns:
        coarse_flag  : xr.DataArray, same dims/coords as coarse_depth
    """
    # Convert to numpy
    fm = fine_mask.values.astype(bool)
    fd = fine_depth.values.astype(np.float32)
    cd = coarse_depth.values.astype(np.float32)

    zf, yf, xf = fd.shape
    zc, yc, xc = cd.shape
    if jmap.shape != (yf, xf) or imap.shape != (yf, xf):
        raise ValueError("jmap/imap must have shape (yf, xf) matching fine grid.")

    # Normalize depth sign to positive-down
    #def _pos_down(depth):
    #    grad = np.nanmedian(np.diff(depth.reshape(zf, -1), axis=0))
    #    return depth if grad >= 0 else -depth

    #fd = _pos_down(fd)
    #cd = _pos_down(cd)

    # Compute bottom depth of fine grid at each horizontal location
    wet_depth = np.where(fm, fd, -np.inf)   # Mask land levels
    fine_bottom = wet_depth.max(axis=0)     # (yf, xf) max depth per column

    # Scatter max bottom depth to coarse grid
    coarse_bottom = np.full((yc, xc), -np.inf, dtype=np.float32)
    np.maximum.at(coarse_bottom, (jmap.ravel(), imap.ravel()), fine_bottom.ravel())

    # Flag all coarse levels shallower than coarse_bottom
    coarse_flag = (coarse_bottom[None, :, :] >= cd).astype(np.int8)  # (zc, yc, xc)
    coarse_flag = np.array(coarse_flag, copy=True)
    nz, ny, nx = coarse_flag.shape

    #for _ in range(iterations):
    for k in range(nz):
        # Apply maximum filter over horizontal plane
        coarse_flag[k] = maximum_filter(coarse_flag[k], size=3, mode='nearest')

    # Wrap back into DataArray
    return xr.DataArray(
        coarse_flag,
        dims=coarse_depth.dims,
        coords=coarse_depth.coords,
        name="tmask",
        attrs={"description": "1=wet, 0=land; computed from fine mask/depth"}
    )
def coarse_from_fine_indices2(
    fine_mask: xr.DataArray,
    fine_deptht: xr.DataArray,
    coarse_deptht: xr.DataArray,
    fine_to_coarse_j: np.ndarray,
    fine_to_coarse_i: np.ndarray
) -> xr.DataArray:
    """
    Compute coarse grid wet/dry flags from fine grid mask+deptht,
    accounting for spatially varying vertical grids.

    Parameters
    ----------
    fine_mask : xr.DataArray
        Binary mask (1=sea, 0=land) on fine grid, shape (zf, yf, xf).
    fine_deptht : xr.DataArray
        Depth at fine grid points, same shape as fine_mask.
    coarse_deptht : xr.DataArray
        Depth at coarse grid points, shape (zc, yc, xc).
    fine_to_coarse_j, fine_to_coarse_i : np.ndarray
        Index maps (yf, xf) -> (yc, xc) mapping fine cells to coarse cells.

    Returns
    -------
    coarse_flag : xr.DataArray
        Binary mask (1=wet, 0=land) for coarse grid, shape (zc, yc, xc).
    """

    # --- Type checks
    if not all(isinstance(arr, xr.DataArray) for arr in [fine_mask, fine_deptht, coarse_deptht]):
        raise TypeError("fine_mask, fine_deptht, and coarse_deptht must be xarray.DataArray objects")
    if fine_mask.shape != fine_deptht.shape:
        raise ValueError("fine_mask and fine_deptht must have the same shape")
    if fine_mask.ndim != 3 or coarse_deptht.ndim != 3:
        raise ValueError("All inputs must be 3D arrays (z, y, x)")

    zc, yc, xc = coarse_deptht.shape
    zf, yf, xf = fine_deptht.shape

    if fine_to_coarse_j.shape != (yf, xf) or fine_to_coarse_i.shape != (yf, xf):
        raise ValueError("fine_to_coarse_j/i must match the horizontal fine grid shape (yf, xf)")

    # --- Compute wet depths at fine grid points
    fine_wet_depth = fine_mask.values * fine_deptht.values  # (zf, yf, xf)

    # --- Flatten horizontal dimensions
    fine_wet_depth_flat = fine_wet_depth.reshape(zf, -1)
    j_flat = fine_to_coarse_j.ravel()
    i_flat = fine_to_coarse_i.ravel()

    # --- Initialize output
    coarse_flag = np.zeros_like(coarse_deptht.values, dtype=np.int8)

    # --- Loop over depth levels
    for k in range(zc):
        # Broadcast coarse depth level to fine cells (mapped indices)
        coarse_depth_at_fine = coarse_deptht.values[k, j_flat, i_flat]  # (yf*xf,)

        # Compare fine wet depth to coarse depth
        is_wet = fine_wet_depth_flat <= coarse_depth_at_fine[np.newaxis, :] & fine_wet_depth_flat > 0
        wet_any = np.any(is_wet, axis=0)  # (yf*xf,)

        # Reduce back to coarse grid
        for idx, (jj, ii) in enumerate(zip(j_flat, i_flat)):
            if wet_any[idx]:
                coarse_flag[k, jj, ii] = 1

    # --- Wrap in DataArray
    coarse_flag_da = xr.DataArray(
        coarse_flag,
        dims=coarse_deptht.dims,
        coords=coarse_deptht.coords,
        name="coarse_flag",
        attrs={"description": "1 if coarse cell is wet, 0 if dry"}
    )

    return coarse_flag_da

def infill_to_fine_mask(arr_in, coarse_ocean_needed):
    """
    Infill coarse grid NaNs only where needed for regridding to a fine mesh.

    Args:
        arr_in (ndarray): Coarse field with NaNs over land (2D).
        coarse_bathy (ndarray): Coarse bathymetry mask (1 for ocean, 0 for land).
        fine_mask (ndarray): Boolean fine-grid ocean mask.
        coarse_from_fine (callable): Function or regridder that maps fine->coarse.
                                     Should accept fine_mask and return coarse coverage.

    Returns:
        ndarray: Coarse field with NaNs filled where needed.
    """
    if arr_in.ndim != 2:
        raise ValueError("Input array must be 2D")

    # Determine which coarse cells influence fine ocean cells
    #coarse_ocean_needed = coarse_from_fine(fine_mask) > 0

    # Mask of NaNs that *need* to be filled
    fill_mask = np.isnan(arr_in) & coarse_ocean_needed 
    
    
    mask_np = np.asarray(fill_mask, dtype=bool)  # Convert xarray/dask to plain ndarray
    mask_np = np.squeeze(mask_np)               # Remove singleton dimensions

    assert mask_np.ndim == 2, f"Mask is not 2D: {mask_np.shape}"
    # Distance transform to get "fill order"
    distance = ndimage.distance_transform_edt(~mask_np)
    arr_filled = arr_in.copy()
    jpj, jpi = arr_filled.shape

    # Sort NaN cells by distance (BFS order)
    nan_indices = np.argwhere(mask_np)
    distances = distance[mask_np]
    order = np.argsort(distances)
    nan_indices = nan_indices[order]
    # Fill progressively
    for y, x in nan_indices:
        neighbors = []
        for dy, dx in [(-1,0),(1,0),(0,-1),(0,1)]:
            ny, nx = y+dy, x+dx
            if 0 <= ny < jpj and 0 <= nx < jpi:
                val = arr_filled[ny, nx]
                if not np.isnan(val):
                    neighbors.append(val)
        if neighbors:
            arr_filled[y, x] = np.nanmean(neighbors)

    return arr_filled

def infill(arr_in, n_iter=None, bathy=None, periodic=False):
    """
    Returns data with any NaNs replaced by iteratively taking the geometric
    mean of surrounding points until all NaNs are removed or n_inter-ations
    have been performed. Input data must be 2D and can include a
    bathymetry array as to provide land barriers to the infilling.

    Args:
        arr_in          (ndarray): data array 2D
        n_iter              (int): number of smoothing iterations
        bathy           (ndarray): bathymetry array (land set to zero)

    Returns:
        arr_mod         (ndarray): modified data array
    """

    # Check number of dims
    if arr_in.ndim != 2:
        raise ValueError("Array must have two dimensions")

    # Intial setup to prime things for the averaging loop
    if bathy is None:
        bathy = np.ones_like(arr_in, dtype=float)
    if n_iter is None:
        n_iter = np.inf
    ind = np.where(np.logical_and(np.isnan(arr_in), np.greater(bathy, 0.)))
    counter = 0
    jpj, jpi = arr_in.shape
    
    arr_in = np.where(bathy>0.5, arr_in, np.nan)
    
    n_nan_n = len(ind[0])
    n_nan_b = len(ind[0]) + 1
    
    # Infill until all NaNs are removed or N interations completed
    while len(ind[0])>0 and n_nan_n<n_nan_b and counter<n_iter:

        # TODO: include a check to see if number of NaNs is decreasing
        n_nan_b = n_nan_n
        
        # Create indices of neighbouring points
        ind_e = cp.deepcopy(ind); ind_w = cp.deepcopy(ind)
        ind_n = cp.deepcopy(ind); ind_s = cp.deepcopy(ind)

        if periodic:
            ind_e[1][:] = ind_e[1][:]+1
            ind_e[1][:] = np.where(ind_e[1][:]==jpi, 0, ind_e[1][:])
            ind_w[1][:] = ind_w[1][:]-1
            ind_w[1][:] = np.where(ind_w[1][:]==-1, jpi-1, ind_w[1][:])
        else:   
            ind_e[1][:] = np.minimum(ind_e[1][:]+1, jpi-1)
            ind_w[1][:] = np.maximum(ind_w[1][:]-1, 0    )
        ind_n[0][:] = np.minimum(ind_n[0][:]+1, jpj-1)
        ind_s[0][:] = np.maximum(ind_s[0][:]-1, 0    )

        # Replace NaNs
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            arr_in[ind] = np.nanmean(np.vstack((arr_in[ind_e],
                                                arr_in[ind_w],
                                                arr_in[ind_n],
                                                arr_in[ind_s])), axis=0)

        # Find new indices for next loop
        ind = np.where(np.logical_and(np.isnan(arr_in),
                                      np.greater(bathy, 0.)))
        counter += 1
        
        n_nan_n = len(ind[0])
        
    return arr_in

def vertical_fill(filled_da, lsm_mask):
    """
    Fill NaNs in 3D array by copying from previous depth level,
    only where lsm_mask indicates ocean.

    Args:
        filled_da (np.ndarray): 3D array (nz, ny, nx) with NaNs.
        lsm_mask (np.ndarray): 3D array (nz, ny, nx), 1=ocean, 0=land.

    Returns:
        np.ndarray: Filled array.
    """
    out = filled_da.copy()
    out_np = out.values
    
    nz, ny, nx = out.shape

    for k in range(1, nz):  # start from 1 (since k-1 needed)
        mask = np.isnan(out_np[k]) & (lsm_mask[k] == 1)

        if np.any(mask):
            out_np[k][mask] = out_np[k-1][mask]

    out = xr.DataArray(
                        da.from_array(out_np, chunks=(1, -1, -1)), dims=out.dims, coords=out.coords, attrs=out.attrs
                    )
    return out

@njit(parallel=True)
def _vertical_interp_kernel(
    src_data, src_depth, dst_depth,
    jmap, imap, fill_value
):
    """
    Core interpolation kernel with parallelization over fine grid points.
    """
    fns, fny, fnx = dst_depth.shape

    dst_data = np.full((fns, fny, fnx), fill_value, dtype=src_data.dtype)

    for jf in prange(fny):
        for if_ in range(fnx):
            jc = jmap[jf, if_]
            ic = imap[jf, if_]

            # Skip invalid mapping
            if jc < 0 or ic < 0:
                continue

            src_z = src_depth[:, jc, ic]
            src_v = src_data[:, jf, if_]
            dst_z = dst_depth[:, jf, if_]

            # Mask NaNs
            valid = np.isfinite(src_v) & np.isfinite(src_z)
            if np.sum(valid) < 2:
                continue

            z_valid = src_z[valid]
            v_valid = src_v[valid]

            # Sort depths
            order = np.argsort(z_valid)
            z_valid = z_valid[order]
            v_valid = v_valid[order]

            # Interpolate
            for k in range(fns):
                ztgt = dst_z[k]
                if np.isnan(ztgt):
                    dst_data[k, jf, if_] = fill_value
                    continue
                if ztgt <= z_valid[0]:
                    dst_data[k, jf, if_] = v_valid[0]
                elif ztgt >= z_valid[-1]:
                    dst_data[k, jf, if_] = v_valid[-1]
                else:
                    # Manual linear interpolation
                    idx = np.searchsorted(z_valid, ztgt) - 1
                    z0, z1 = z_valid[idx], z_valid[idx + 1]
                    v0, v1 = v_valid[idx], v_valid[idx + 1]
                    dst_data[k, jf, if_] = v0 + (v1 - v0) * (ztgt - z0) / (z1 - z0)

    return dst_data


def vertical_regrid_with_index_map(
    src_data: xr.DataArray,
    src_depth: xr.DataArray,
    dst_depth: xr.DataArray,
    jmap: np.ndarray,
    imap: np.ndarray,
    fill_value=np.nan,
) -> xr.DataArray:
    """
    Vertically interpolates source data to destination depth grid
    using precomputed coarse->fine index maps.

    Args:
        src_data:   DataArray (nz, ny, nx)
        src_depth:  DataArray (nz, ny, nx)
        dst_depth:  DataArray (fns, fny, fnx)
        jmap:       np.ndarray (fny, fnx) - coarse y index for each fine cell
        imap:       np.ndarray (fny, fnx) - coarse x index for each fine cell
        fill_value: value to fill outside valid range

    Returns:
        DataArray of shape (fns, fny, fnx)
    """
    src_data_np = np.asarray(src_data)
    src_depth_np = np.asarray(src_depth)
    dst_depth_np = np.asarray(dst_depth)

    dst_np = _vertical_interp_kernel(
        src_data_np, src_depth_np, dst_depth_np, jmap, imap, fill_value
    )

    return xr.DataArray(
        dst_np,
        dims=("z", "y", "x"),
        name=src_data.name if src_data.name else "regridded_var",
    )

def enforce_static_stability_vectorized(
    toce_filled_da: xr.DataArray,
    soce_filled_da: xr.DataArray,
    depth_da: xr.DataArray,
) -> tuple[xr.DataArray, xr.DataArray]:
    """
    Ensure static stability in the vertical by copying unstable layers
    from the layer above. Fully vectorized NumPy implementation.

    Args:
        toce_filled_da : Conservative temperature (nz, ny, nx)
        soce_filled_da : Absolute salinity (nz, ny, nx)
        depth_da       : Depth array (nz, ny, nx)

    Returns:
        xr.DataArray : Stable Conservative temperature
        xr.DataArray : Stable Absolute salinity
    """

    # Convert to numpy
    toce = toce_filled_da.values.copy()
    soce = soce_filled_da.values.copy()
    depth = depth_da.values

    nz, ny, nx = toce.shape

    # Compute density for all levels
    rho = np.empty_like(toce)
    for k in range(nz):
        rho[k] = compute_density(soce[k], toce[k], depth[k])

    # Build stability mask: unstable if rho[k] < rho[k-1]
    unstable = rho[1:] < rho[:-1]   # shape (nz-1, ny, nx)

    # Propagate corrections downward
    for k in range(1, nz):
        mask = unstable[k-1]
        if np.any(mask):
            toce[k][mask] = toce[k-1][mask]
            soce[k][mask] = soce[k-1][mask]
            rho[k][mask]  = rho[k-1][mask]  # keep density consistent

    # Wrap back into DataArrays
    toce_out = xr.DataArray(
        da.from_array(toce, chunks=(1, -1, -1)), dims=toce_filled_da.dims, coords=toce_filled_da.coords, name=toce_filled_da.name
    )
    soce_out = xr.DataArray(
        da.from_array(soce, chunks=(1, -1, -1)), dims=soce_filled_da.dims, coords=soce_filled_da.coords, name=soce_filled_da.name
    )

    return toce_out, soce_out


def compute_density(SA, CT, depth):
    """Compute in-situ density (kg/m³) from SA, CT, and depth (m)."""
    # Pressure in dbar ~ depth (m) * 0.1
    p = depth * 0.1
    return gsw.rho(SA, CT, p)

@njit(parallel=True)
def _vertical_interp_kernel(
    src_data, src_depth, dst_depth,
    jmap, imap, fill_value
):
    """
    Core interpolation kernel with parallelization over fine grid points.
    """
    fns, fny, fnx = dst_depth.shape

    dst_data = np.full((fns, fny, fnx), fill_value, dtype=src_data.dtype)

    for jf in prange(fny):
        for if_ in range(fnx):
            jc = jmap[jf, if_]
            ic = imap[jf, if_]

            # Skip invalid mapping
            if jc < 0 or ic < 0:
                continue

            src_z = src_depth[:, jc, ic]
            src_v = src_data[:, jf, if_]
            dst_z = dst_depth[:, jf, if_]

            # Mask NaNs
            valid = np.isfinite(src_v) & np.isfinite(src_z)
            if np.sum(valid) < 2:
                continue

            z_valid = src_z[valid]
            v_valid = src_v[valid]

            # Sort depths
            order = np.argsort(z_valid)
            z_valid = z_valid[order]
            v_valid = v_valid[order]

            # Interpolate
            for k in range(fns):
                ztgt = dst_z[k]
                if np.isnan(ztgt):
                    dst_data[k, jf, if_] = fill_value
                    continue
                if ztgt <= z_valid[0]:
                    dst_data[k, jf, if_] = v_valid[0]
                elif ztgt >= z_valid[-1]:
                    dst_data[k, jf, if_] = v_valid[-1]
                else:
                    # Manual linear interpolation
                    idx = np.searchsorted(z_valid, ztgt) - 1
                    z0, z1 = z_valid[idx], z_valid[idx + 1]
                    v0, v1 = v_valid[idx], v_valid[idx + 1]
                    dst_data[k, jf, if_] = v0 + (v1 - v0) * (ztgt - z0) / (z1 - z0)

    return dst_data

def vertical_regrid_with_index_map(
    src_data: xr.DataArray,
    src_depth: xr.DataArray,
    dst_depth: xr.DataArray,
    jmap: np.ndarray,
    imap: np.ndarray,
    fill_value=np.nan,
) -> xr.DataArray:
    """
    Vertically interpolates source data to destination depth grid
    using precomputed coarse->fine index maps.

    Args:
        src_data:   DataArray (nz, ny, nx)
        src_depth:  DataArray (nz, ny, nx)
        dst_depth:  DataArray (fns, fny, fnx)
        jmap:       np.ndarray (fny, fnx) - coarse y index for each fine cell
        imap:       np.ndarray (fny, fnx) - coarse x index for each fine cell
        fill_value: value to fill outside valid range

    Returns:
        DataArray of shape (fns, fny, fnx)
    """
    src_data_np = np.asarray(src_data)
    src_depth_np = np.asarray(src_depth)
    dst_depth_np = np.asarray(dst_depth)

    dst_np = _vertical_interp_kernel(
        src_data_np, src_depth_np, dst_depth_np, jmap, imap, fill_value
    )

    return xr.DataArray(
        dst_np,
        dims=("z", "y", "x"),
        name=src_data.name if src_data.name else "regridded_var",
    )


### Specify source grid

NB we remove last two columns as this is output from NEMO<4.2

In [11]:
# source data
src_fname = '/scratch/benbar/ARC36/Initial_Conditions/NPD025_Data/eORCA025_1m_grid_T_200806-200806.nc'
# source grid
src_dom = '/scratch/benbar/ARC36/Initial_Conditions/domain_cfg_eORCA025.nc'

In [12]:
# create dataset
ds_src_grid = xr.open_dataset(src_dom, chunks={}).isel(t=0, y=slice(0, -2)).rename({'glamt': 'lon', 'gphit': 'lat'})
ds_src_grid = ds_src_grid.chunk({'z': 9, 'y': 603, 'x': 720}) # adjust chunk sizes to be 10 to 100 mb
ds_src_grid = ds_src_grid.set_coords(("lat", "lon"))

# create dataset
ds_src_data = xr.open_dataset(src_fname).isel(y=slice(0, -2)).rename({'nav_lon': 'lon', 'nav_lat': 'lat'})
ds_src_data = ds_src_data.chunk({'deptht': 9, 'y': 603, 'x': 720}) # adjust chunk sizes to be 10 to 100 mb


### Specify destination grid

In [13]:
# destination grid
dst_dom = '/scratch/benbar/NAARC/domain_cfg_mes.nc'
out_folder = '/scratch/benbar/NAARC/Initial_Conditions/'

# create dataset and add missing coordinates
ds_dst_grid = xr.open_dataset(dst_dom, chunks={}).isel(time_counter=0).rename({'xx': 'lon', 'y': 'lat'})
ds_dst_grid = ds_dst_grid.chunk({'nav_lev': 3, 'lat': 721, 'lon': 1080})
ds_dst_grid = ds_dst_grid.assign_coords(
    lon=(("y", "x"), ds_dst_grid["glamt"].values), # glamt
    lat=(("y", "x"), ds_dst_grid["gphit"].values) # gphit
)

### Derive mapping indices from fine to coarse grid

Identify which cells on the coarse grid need to be infilled before regridding to the finer grid

In [14]:
j_map, i_map = compute_index_map_kdtree(ds_dst_grid, ds_src_grid)

### Derive LSMs and 3D depth arrays for each grid

NB we don't have a domain_cfg for the source grid so use input data as a proxy 

In [15]:
%%time
# derive tmask information to source grid
ds_src_grid = add_tmask_from_field(ds_src_data.thetao_con, ds_src_grid, missing_value=0.0)

CPU times: user 9.13 ms, sys: 936 μs, total: 10.1 ms
Wall time: 46.9 ms


In [16]:
%%time
# derive depth information to source grid
ds_src_grid['gdept'], _ = e3_to_depth(ds_src_grid.e3t_0, ds_src_grid.e3w_0, 75)
#ds_src_grid["gdept"] = ds_src_data.deptht.broadcast_like(ds_src_data.votemper[0,:,:,:].squeeze())

CPU times: user 4min 38s, sys: 20.9 s, total: 4min 59s
Wall time: 5min 34s


In [17]:
%%time
# derive tmask information to destination grid 
ds_dst_grid = add_tmask_domcfg(ds_dst_grid)

CPU times: user 51.8 s, sys: 55.4 ms, total: 51.8 s
Wall time: 52 s


In [18]:
%%time
# derive depth information to destination grid 
if 1:
    ds_dst_grid["gdept"], _ = e3_to_depth(ds_dst_grid.e3t_0, ds_dst_grid.e3w_0, 75)
    ds_dst_grid["gdept"].to_netcdf(out_folder + '/naarc_gdept.nc')
else:
    ds_dst_gdept = xr.open_dataset(out_folder + '/naarc_gdept.nc')  
    ds_dst_grid["gdept"] = ds_dst_gdept["gdept"]

CPU times: user 3min 25s, sys: 1min 1s, total: 4min 27s
Wall time: 9min 52s


### Derive a tmask of the fine grid on the coarse grid

This allows the infilling of only points required when regridding

In [19]:
%%time
dst_on_src_lsm = coarse_from_fine_indices3(ds_dst_grid.tmask, ds_dst_grid.gdept, ds_src_grid.gdept, j_map, i_map) 

CPU times: user 4min 50s, sys: 48.4 s, total: 5min 39s
Wall time: 9min 7s


### Produce updated IC fields on coarse grid

In [20]:
%%time
time = 0 # only pulling out first time-slice
results = []
for k in range(75):
    var_slice = ds_src_data.thetao_con[time,k,:,:].squeeze().values
    var_slice = np.where(var_slice==0, np.nan, var_slice)
    tmplsm = dst_on_src_lsm[k,:,:].squeeze().values==1
    fill_mask = np.isnan(var_slice) & tmplsm
    arr_in = np.where(fill_mask, np.nan, var_slice)
    bat_in = np.where(tmplsm, 1, 0)
    filled = infill(arr_in, n_iter=None, bathy=bat_in, periodic=True)
    results.append(filled)

# Combine into one 3D NumPy array
filled_all = np.stack(results, axis=0)  # shape (75, ny, nx)

# Wrap back into DataArray with correct dims/coords
filled_da_toce = xr.DataArray(
    da.from_array(filled_all, chunks=(1, -1, -1)),
    dims=("deptht", "y", "x"),
    coords={
        "deptht": ds_src_data.deptht.isel(deptht=slice(0,75)),
        "y": ds_src_data.y,
        "x": ds_src_data.x,
    },
    name="toce"
)

CPU times: user 25.9 s, sys: 394 ms, total: 26.3 s
Wall time: 31.8 s


In [21]:
%%time
results = []
for k in range(75):
    var_slice = ds_src_data.so_abs[time,k,:,:].squeeze().values
    var_slice = np.where(var_slice==0, np.nan, var_slice)
    tmplsm = dst_on_src_lsm[k,:,:].squeeze().values==1
    fill_mask = np.isnan(var_slice) & tmplsm
    arr_in = np.where(fill_mask, np.nan, var_slice)
    bat_in = np.where(tmplsm, 1, 0)
    filled = infill(arr_in, n_iter=None, bathy=bat_in, periodic=True)
    results.append(filled)

# Combine into one 3D NumPy array
filled_all = np.stack(results, axis=0)  # shape (75, ny, nx)

# Wrap back into DataArray with correct dims/coords
filled_da_soce = xr.DataArray(
    da.from_array(filled_all, chunks=(1, -1, -1)),
    dims=("deptht", "y", "x"),
    coords={
        "deptht": ds_src_data.deptht.isel(deptht=slice(0,75)),
        "y": ds_src_data.y,
        "x": ds_src_data.x,
    },
    name="soce"
)

CPU times: user 25.7 s, sys: 349 ms, total: 26.1 s
Wall time: 29.1 s


In [22]:
%%time
# downfill for any isolated points
filled_da_soce_ud = vertical_fill(filled_da_soce,dst_on_src_lsm)
filled_da_toce_ud = vertical_fill(filled_da_toce,dst_on_src_lsm)

CPU times: user 2.21 s, sys: 329 ms, total: 2.54 s
Wall time: 2.55 s


In [23]:
%%time
# ensure water column is statically stable
filled_da_toce_ud_df, filled_da_soce_ud_df = enforce_static_stability_vectorized(filled_da_toce_ud,filled_da_soce_ud,ds_src_grid.gdept)

CPU times: user 2min 31s, sys: 10.4 s, total: 2min 42s
Wall time: 2min 26s


### Produce xESMF regridder

This can be saved and reused saving CPU time

In [24]:
ds_src_grid = ds_src_grid.assign_coords(
    lon=(("y", "x"), ds_src_grid["lon"].values), # glamt
    lat=(("y", "x"), ds_src_grid["lat"].values), # gphit
    deptht=("z", ds_src_grid["deptht"].values),
#    tmask=(("z", "y", "x"), ds_src_grid["tmask"].values)
)

In [25]:
# make tmask z based
ds_src_grid["tmask"] = ds_src_grid["tmask"].rename({"deptht": "z"})

In [26]:
ds_dst_grid = ds_dst_grid.assign_coords(
    lon=(("y", "x"), ds_dst_grid["lon"].values), # glamt
    lat=(("y", "x"), ds_dst_grid["lat"].values), # gphit
    deptht=("z", ds_dst_grid["nav_lev"].values),
#    tmask=(("z", "y", "x"), ds_dst_grid["tmask"].values)
)

In [27]:
vars_deptht = [v for v in ds_dst_grid.data_vars if "nav_lev" in ds_dst_grid[v].dims]
for v in vars_deptht:
    ds_dst_grid[v] = ds_dst_grid[v].rename({"nav_lev": "z"})
    

In [28]:
ds_dst_grid = ds_dst_grid.drop_vars('nav_lev')

In [29]:
# make tmask not a coord
#ds_dst_grid = ds_dst_grid.reset_coords("tmask")
#ds_dst_grid["tmask"] = ds_dst_grid["tmask"].swap_dims({"lat": "y", "lon": "x"})

In [30]:
vars_deptht = [v for v in ds_dst_grid.data_vars if "lat" in ds_dst_grid[v].dims]
for v in vars_deptht:
    ds_dst_grid[v] = ds_dst_grid[v].rename({"lat": "y"})
    
vars_deptht = [v for v in ds_dst_grid.data_vars if "lon" in ds_dst_grid[v].dims]
for v in vars_deptht:
    ds_dst_grid[v] = ds_dst_grid[v].rename({"lon": "x"})

### Data is too large to make the regridder so coarsen it for interpolation then remap it later

In [31]:
%%time
subset = 0
if subset:
    ds_dst_grid_c = ds_dst_grid.isel(
        x=slice(None, None, 4),
        y=slice(None, None, 4),
    )
else:
    nx = ds_dst_grid.sizes["x"]
    ny = ds_dst_grid.sizes["y"]
    x_step = nx // 4
    y_step = ny // 4

    for iy in range(4):
        for ix in range(4):
            print(iy, ix)
            x0 = ix * x_step
            x1 = nx if ix == 3 else (ix + 1) * x_step
            y0 = iy * y_step
            y1 = ny if iy == 3 else (iy + 1) * y_step

            ds_dst_grid_c = ds_dst_grid.isel(
                x=slice(x0, x1),
                y=slice(y0, y1),
            )
            ds_dst_grid_c = ds_dst_grid_c.chunk({'z': 3, 'y': 272, 'x': 1080})

            # create the regridding weights
            regridder = xe.Regridder(ds_src_grid.squeeze(), ds_dst_grid_c, method="nearest_s2d", extrap_method='nearest_s2d', periodic=True, ignore_degenerate=True)

            regridded_toce = xr.zeros_like(ds_dst_grid_c.gdept).rename("toce")
            regridded_soce = xr.zeros_like(ds_dst_grid_c.gdept).rename("soce")
            depth_values = ds_src_data.deptht.values
            for k in np.arange(0,len(depth_values),1):
                arr2d = filled_da_toce_ud_df.isel(deptht=k)
                regridded_toce[k] = regridder(arr2d.values, skipna=True)
                arr2d = filled_da_soce_ud_df.isel(deptht=k)
                regridded_soce[k] = regridder(arr2d.values, skipna=True)
                
            # temperature onto final vertical grid
            toce_final = vertical_regrid_with_index_map(regridded_toce, ds_src_grid.gdept, ds_dst_grid_c.gdept, j_map, i_map)
            toce_final = xr.where(ds_dst_grid_c.tmask.values>0.5, toce_final, np.nan)
            # salinity onto final vertical grid
            soce_final = vertical_regrid_with_index_map(regridded_soce, ds_src_grid.gdept, ds_dst_grid_c.gdept, j_map, i_map)
            soce_final = xr.where(ds_dst_grid_c.tmask.values>0.5, soce_final, np.nan)
            
            # combine into a Dataset
            ds_out = xr.Dataset(
                {
                    "toce": toce_final,
                    "soce": soce_final,
                }
            )

            # replace NaNs with fill value
            fill_value = 1e20
            ds_out = ds_out.fillna(fill_value)

            encoding = {
                var: {
                    "zlib": True,
                    "complevel": 1,
                    "_FillValue": fill_value,
                    "dtype": "float32",
                    "chunksizes": (1, 721, 1080),  # (deptht, y, x) chunk shape on disk
                }
                for var in ds_out.data_vars
            }

            ds_out.to_netcdf(
                out_folder + "IC_compressed_" + str(iy) + "_" + str(ix) + ".nc",
                mode="w",
                format="NETCDF4",
                engine="netcdf4",
                encoding=encoding,
            )
            

0 0
0 1
0 2
0 3
1 0
1 1
1 2
1 3
2 0
2 1
2 2
2 3
3 0
3 1
3 2
3 3
CPU times: user 1h 46min 54s, sys: 7min 3s, total: 1h 53min 57s
Wall time: 1h 38min 20s
